# Eksploracja danych  - PS 3 - Tomasz Jarmoc i Krzysztof Sobolewski


Dla zbioru danych użytego w raporcie 3 (do klasyfikacji) przeprowadź analizę skupień wyłączając kolumnę ze zmienną zależną (lub etykietami klasy, jeśli to zbiór do klasyfikacji). W drugim przypadku użyj kolumny z etykietami klasy do sprawdzenia jakości uzyskanych klastrów.

Użyj:

- metod hierarchicznych (z 3 różnymi sposobami liczenia odległości między klastrami) - sporządź dendrogram dla każdej z metod
- metody k-średnich
- metody EM
- metody DBSCAN
- jednej innej - dowolnej metody

Porównaj uzyskane wyniki - liczbę uzyskanych klastrów i ich jakość.

In [ ]:
import os
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, Birch, DBSCAN, KMeans
from sklearn.compose import ColumnTransformer
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

plik_csv = 'insurance.csv'
link_do_pliku = 'https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv'
if not os.path.exists(plik_csv):
    urlretrieve(link_do_pliku, plik_csv)

df = pd.read_csv(plik_csv)
print(f'Wymiary danych: {df.shape}')
display(df.head())


In [ ]:
# smoker jest etykiet? do oceny, a nie cech? wykorzystywan? do grupowania.
y = df['smoker']
X = df.drop(columns='smoker')
kolumny_numeryczne = X.select_dtypes(include=np.number).columns.tolist()
kolumny_kategoryczne = X.select_dtypes(exclude=np.number).columns.tolist()

przetwarzanie = ColumnTransformer(transformers=[
    ('num', StandardScaler(), kolumny_numeryczne),
    ('kat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), kolumny_kategoryczne),
])
X_scaled = przetwarzanie.fit_transform(X)
y_encoded = (y == 'yes').astype(int)
print(f'Liczba cech po kodowaniu: {X_scaled.shape[1]}')


## 1. Metody hierarchiczne

Poni?ej por?wnujemy dendrogramy dla metod Ward, complete i average. Dla czytelno?ci pokazane jest ostatnie 30 po??cze?.

In [ ]:
metody_linkage = ['ward', 'complete', 'average']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metoda in zip(axes, metody_linkage):
    Z = linkage(X_scaled, method=metoda, metric='euclidean')
    dendrogram(Z, truncate_mode='lastp', p=30, ax=ax)
    ax.set_title(f'Dendrogram: {metoda}')
    ax.set_xlabel('Klastry (uci?ty dendrogram)')
    ax.set_ylabel('Odleg?o??')

plt.tight_layout()
display(fig)
plt.close(fig)


## 2. Pozosta?e metody klasteryzacji

Dla por?wnywalno?ci ustalamy k=2 w metodach wymagaj?cych liczby klastr?w. DBSCAN sam wyznacza klastry; etykieta -1 oznacza szum.

In [ ]:
wyniki = {}

for metoda in metody_linkage:
    model = AgglomerativeClustering(n_clusters=2, linkage=metoda)
    wyniki[f'HAC ({metoda})'] = model.fit_predict(X_scaled)

wyniki['K-?rednich'] = KMeans(n_clusters=2, random_state=42, n_init=10).fit_predict(X_scaled)
wyniki['EM (Gaussian Mixture)'] = GaussianMixture(n_components=2, random_state=42).fit_predict(X_scaled)
wyniki['DBSCAN'] = DBSCAN(eps=1.5, min_samples=10).fit_predict(X_scaled)
wyniki['Birch'] = Birch(n_clusters=2).fit_predict(X_scaled)

print(f'Uruchomiono {len(wyniki)} wariant?w klasteryzacji.')


## 3. Por?wnanie jako?ci

Silhouette ocenia sp?jno?? i separacj? klastr?w. ARI i NMI mierz? zgodno?? z etykiet? smoker, kt?ra nie by?a u?yta do grupowania.

In [ ]:
wiersze = []
for nazwa, etykiety in wyniki.items():
    maska = etykiety != -1
    klastry = np.unique(etykiety[maska])
    silhouette = np.nan
    if len(klastry) >= 2 and maska.sum() > len(klastry):
        silhouette = silhouette_score(
            X_scaled[maska], etykiety[maska],
            sample_size=min(1000, int(maska.sum())), random_state=42,
        )
    wiersze.append({
        'Metoda': nazwa,
        'Liczba klastr?w': len(klastry),
        'Szum (DBSCAN)': int((etykiety == -1).sum()),
        'Silhouette': silhouette,
        'ARI wzgl?dem smoker': adjusted_rand_score(y_encoded, etykiety),
        'NMI wzgl?dem smoker': normalized_mutual_info_score(y_encoded, etykiety),
    })

porownanie = pd.DataFrame(wiersze).set_index('Metoda')
display(porownanie.round(3))


## 4. Wykresy por?wnawcze

Wykresy pokazuj? liczb? znalezionych klastr?w, Silhouette oraz zgodno?? klastr?w z etykiet? smoker.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

porownanie['Liczba klastr?w'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Liczba klastr?w')
axes[0].set_ylabel('Liczba')

porownanie['Silhouette'].dropna().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Silhouette (wy?ej = lepiej)')

porownanie[['ARI wzgl?dem smoker', 'NMI wzgl?dem smoker']].plot(kind='bar', ax=axes[2])
axes[2].set_title('Zgodno?? z etykiet? smoker')
axes[2].set_ylabel('Wynik')

for ax in axes:
    ax.tick_params(axis='x', labelrotation=60)
    ax.grid(axis='y', alpha=0.25)

plt.tight_layout()
display(fig)
plt.close(fig)
